# Step 2. Initial conditions: CLIM_BASE state + DART perturbations (FREECLIM_DARTPERT20)

## Purpose

Builds the 20-member CLIM_BASE initial ensemble: the free-running `F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM` state of each start date (atmosphere and all other components) with DART-derived atmospheric perturbations. The reference is a full EAM initial file, so no completion step is needed.

## Workflow

```text
step 1  01_select_dart_20_member_subset.ipynb   20 DART members per season
          |
          +--> step 2  02_ic_free_clim_dartpert.ipynb      CLIM_BASE state          + DART perturbations -> FREECLIM_DARTPERT20
          +--> step 3  03_ic_era5_dartfill_dartpert.ipynb  ERA5 + DART-mean fields  + DART perturbations -> HYB_DARTPERT20C
          +--> step 4  04_ic_era5_climfill_dartpert.ipynb  ERA5 + CLIM_BASE fields  + DART perturbations -> HYB_CLIMFILL_DARTPERT20
```

## Method

For the 20 selected DART members $S$ of the season (step 1) and member $s_i \in S$:

$$\delta A_{s_i} = A_{\mathrm{DART},s_i} - \overline{A}_{\mathrm{DART},S}, \qquad
A_{\mathrm{EN}(i+1)} = A_{\mathrm{ref}} + \alpha\,\delta A_{s_i},\ i = 1,\dots,19, \qquad A_{\mathrm{EN01}} = A_{\mathrm{ref}}.$$

The 20th selected member is in the mean but gives no output member, so the output ensemble mean is offset from the reference by $-\alpha\,\delta A_{\mathrm{omitted}}/20$; the written files are checked against that offset (not enforced for the clipped moisture fields). Only the perturbed fields change: the structure, attributes and every other field of the reference file are checked to be identical. The reference's land, river, sea-ice, coupler restarts and rpointers are linked into each date directory.

The existing library (`free_clim_dartpert20_atm_multidate`) perturbed T, U, V, Q and PS; `PERTURB_VARS` keeps that, so a rerun validates and reuses it. Adding `H2OLNZ` (the radiative water vapour) would let the radiation see each member's moisture perturbation, as in steps 3 and 4, but changes the library: rebuild with `OVERWRITE = True` into a new `OUTPUT_ROOT`.

## Outputs

Each date directory `<OUTPUT_ROOT>/<date>/` holds `<OUT_PREFIX>.EN01-EN20.eam.i.<date>.nc`, the component restart links, `output_to_dart_member_mapping.csv`, `perturbation_diagnostics.csv`, `q_clipping_diagnostics.csv`, `written_file_validation.csv` and `dart_perturbation_metadata.json`; `<OUTPUT_ROOT>/all_*.csv` summarize all dates.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

## 2. User setup

Plain values only, so `papermill -p` overrides work.

In [ ]:
DART_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/e3sm_dart")   # <case>/[EN##/]archive/rest/<date>/
SELECTION_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/dart_subset_selection")  # checked when present
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
SEASONS = None            # None: both; or "dec2011" / "may2012"
OVERWRITE = False         # True rewrites existing outputs; False validates and reuses them

CLIM_CASE = "F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM"
CLIM_REST_ROOT = DART_ROOT / CLIM_CASE / "archive" / "rest"                    # <date>/<case>.eam.i.<date>.nc
OUTPUT_ROOT = DART_ROOT / "free_clim_dartpert20_atm_multidate"
OUT_PREFIX = "FREECLIM_DARTPERT20"
PERTURB_VARS = ["T", "U", "V", "Q", "PS"]   # as built; see Method about H2OLNZ

## 3. Modules and inputs

In [ ]:
pkg = WORK_DIR.resolve() / "ensemble_perturbations"
if str(pkg) not in sys.path:
    sys.path.insert(0, str(pkg))
from util import dart_perturbations as dp

seasons = dp.dart_date_sets(DART_ROOT, SEASONS, SELECTION_ROOT)
settings = dp.Settings(variables=tuple(PERTURB_VARS), overwrite=OVERWRITE)
for s in seasons:
    print(f"[SEASON] {s.name}: {s.case}, dates {s.dates}, members {s.members}")
reference = dp.ReferenceCase(
    name="FREE_CLIM",
    atm_file=str(CLIM_REST_ROOT / "{date}" / f"{CLIM_CASE}.eam.i.{{date}}.nc"),
    rest_dir=str(CLIM_REST_ROOT / "{date}"),          # CLIM_BASE land, river, sea ice and coupler restarts
    output_root=OUTPUT_ROOT, out_prefix=OUT_PREFIX)

## 4. Step 2a. Configure the start dates

One row per start date; every selected DART member is checked against the reference grid and layout.

In [ ]:
configs = dp.build_run_configs(reference, seasons, settings)
for c in configs:
    dp.preflight(c, settings)
display(dp.config_table(configs))

## 5. Step 2b. Build the perturbed ensemble

EN01 is a checksum-verified copy of the reference; EN02-EN20 are perturbed. Existing outputs are validated and reused unless `OVERWRITE`.

In [ ]:
tables = dp.build_library(configs, settings, display=display)
display(tables["diagnostics"].pivot_table(index="date", columns="variable", values="mean_member_rms_perturbation"))

## 6. Verdict

In [ ]:
v = tables["validation"]
clip = tables["q_clipping"]
print(f"[CHECK] {len(configs)} dates, {len(configs) * len(settings.all_members)} member files")
print("[CHECK] ensemble-mean closure:", "pass" if v["passes_tolerance"].all() else "FAIL")
if not clip.empty:
    print(f"[CHECK] max fraction of clipped moisture points: {clip['fraction_q_clipped'].max():.2e}")